Event-based directional prediction on SPY with internal data and ML libraries

Outline
- Fetch data (findata)
- Construct dollar bars (mlfinlab)
- Detect events with CUSUM (mlfinlab)
- Label with triple barrier (mlfinlab)
- Build a compact feature set (pandas)
- Train a baseline classifier and evaluate with time-based splits (mlfinlab PurgedKFold)
- Produce asset_weights and asset_returns
- Call backtest(asset_weights, asset_returns)

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

# Data
from findata.equity_prices import get_equity_prices

# MLFinLab components
from mlfinlab.data_structures.standard_data_structures import get_dollar_bars
from mlfinlab.filters.filters import cusum_filter
from mlfinlab.labeling.labeling import get_events, get_bins
from mlfinlab.util.volatility import get_daily_vol
from mlfinlab.cross_validation.cross_validation import PurgedKFold, ml_cross_val_score

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 50)

1) Fetch SPY data (daily OHLCV)

In [ ]:
start_date = "2015-01-01"
end_date = "2024-12-31"

spy_daily = get_equity_prices(
    tickers=["SPY"],
    start_date=start_date,
    end_date=end_date,
    fields=["Close", "High", "Low", "Volume"],
    frequency="1d",
    auto_adjust=True,
)

print(spy_daily.head())
print(spy_daily.tail())
print("Shape:", spy_daily.shape)

2) Construct dollar bars from SPY daily price/volume
Note: get_dollar_bars expects [date_time, price, volume] data. We approximate using daily close and volume.

In [ ]:
# Prepare input for dollar bars: date_time, price, volume
bars_input = (
    spy_daily[["Close", "Volume"]]
    .rename(columns={"Close": "price", "Volume": "volume"})
    .assign(date_time=lambda df: df.index)
    .loc[:, ["date_time", "price", "volume"]]
)

# Construct dollar bars with a threshold tuned for daily data
bars = get_dollar_bars(
    file_path_or_df=bars_input,
    threshold=200_000_000.0,  # $200mm per bar (approx.)
    batch_size=20_000_000,
    verbose=False,
    to_csv=False,
    output_path=None,
)

# Ensure datetime index for downstream steps
bars = bars.copy()
bars["date_time"] = pd.to_datetime(bars["date_time"]) 
bars = bars.set_index("date_time").sort_index()

print(bars.head())
print(bars.tail())
print("Bars shape:", bars.shape)

3) Volatility target (for triple-barrier widths)

In [ ]:
close_series = bars["close"].copy()

daily_vol = get_daily_vol(close_series, lookback=100)
print(daily_vol.dropna().head())
print("Non-null vol points:", daily_vol.notna().sum())

4) Event detection via CUSUM on bar returns

In [ ]:
ret_series = close_series.pct_change().dropna()

# Use a fixed threshold on returns for simplicity; tune as needed
cusum_threshold = 0.01

t_events = cusum_filter(raw_time_series=ret_series, threshold=cusum_threshold, time_stamps=True)

print("#CUSUM events:", len(t_events))
print("First 5:", list(t_events[:5]))

5) Triple-barrier labeling (no vertical barrier), then bins

In [ ]:
# Align target to bar index and forward-fill
trgt = daily_vol.reindex(close_series.index).fillna(method='ffill')

# Generate events
events = get_events(
    close=close_series,
    t_events=pd.Index(t_events),
    pt_sl=[1.0, 1.0],
    target=trgt,
    min_ret=0.0,
    num_threads=4,
    vertical_barrier_times=False,
    side_prediction=None,
    verbose=False,
)

print("Events head:\n", events.head())
print("Events shape:", events.shape)

# Convert to bins/labels
labels = get_bins(triple_barrier_events=events, close=close_series)
print("Labels head:\n", labels.head())
print("Labels value counts:\n", labels['bin'].value_counts(dropna=False))

6) Compact feature set
- Returns over horizons: 1, 3, 5 bars
- Rolling volatility (20)
- Rolling z-score of close (20)

In [ ]:
# Base series
px = close_series.copy()
ret1 = px.pct_change(1)
ret3 = px.pct_change(3)
ret5 = px.pct_change(5)

vol20 = ret1.rolling(20).std()

# Rolling z-score of price using 20 window
roll_mean = px.rolling(20).mean()
roll_std = px.rolling(20).std()
zscore20 = (px - roll_mean) / roll_std

X = pd.concat(
    {
        "ret_1": ret1,
        "ret_3": ret3,
        "ret_5": ret5,
        "vol_20": vol20,
        "zscore_20": zscore20,
    },
    axis=1,
).dropna()

print(X.head())
print("Features shape:", X.shape)

7) Baseline classifier and time-based CV (PurgedKFold)

In [ ]:
# Align labels to feature index and build binary target
labels_aligned = labels.reindex(X.index).dropna()
X_model = X.loc[labels_aligned.index]
y_raw = labels_aligned["bin"].astype(int)
# Ensure binary {0,1}
y = (y_raw > 0).astype(int)

# Samples info sets aligned to X
samples_info_sets = events["t1"].reindex(labels.index).reindex(X_model.index)

clf = LogisticRegression(max_iter=200, solver="lbfgs")
cv = PurgedKFold(n_splits=5, samples_info_sets=samples_info_sets, pct_embargo=0.01)

cv_scores = ml_cross_val_score(
    classifier=clf,
    X=X_model,
    y=y,
    cv_gen=cv,
    sample_weight_train=None,
    sample_weight_score=None,
    scoring=log_loss,
    require_proba=True,
    n_jobs_score=1,
)

print("Fold log-loss:", cv_scores)
print("Mean log-loss:", np.mean(cv_scores))

8) Fit on all data to produce asset_weights, then compute asset_returns

In [ ]:
# Fit baseline model on all aligned data
final_clf = LogisticRegression(max_iter=200, solver="lbfgs")
final_clf.fit(X_model, y)

# Predicted probability for class 1 (up move)
proba = pd.Series(final_clf.predict_proba(X_model)[:, 1], index=X_model.index, name="p_up")

# Convert to weights in [-1, 1]
weights_series = (2.0 * proba - 1.0).clip(-1.0, 1.0)
asset_weights = weights_series.to_frame("SPY")

# Compute asset returns from dollar-bar closes
bar_rets = close_series.pct_change().rename("SPY").to_frame()
# Align, apply 1-bar execution lag
asset_weights = asset_weights.reindex(bar_rets.index).shift(1).fillna(0.0)
asset_returns = (asset_weights["SPY"] * bar_rets["SPY"]).to_frame("SPY")

print("asset_weights head:\n", asset_weights.head())
print("asset_returns head:\n", asset_returns.head())

9) Run user-defined backtest (expects function `backtest(asset_weights, asset_returns)`)